# IT3212 – Assignment 3 – Task 10: Overall comparison (resubmission)

This notebook produces the new results in Section 10 of the report. It only evaluates models; the data, features and model settings are unchanged.

- **Matched contact budgets (Table 21):** all models ranked by predicted probability, top 10–40% contacted, and a common recall of 0.90 with thresholds chosen on validation.
- **AUC decomposition and within-sex AUC (Table 24):** the Task 5 MLP retrained with unchanged settings on feature subsets, as reference models only.
- **Prediction timing (Table 23):** all models on the same CPU, ten repeats.

The five models are rebuilt with their final settings from Tasks 5 and 7 (their test AUC matches the report). The DNN probabilities come from `task9_robustness_probabilities.npz`, written by section 11 of the Task 9 notebook.

In [1]:
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

DATA_DIR = Path("../data/assignment3")
with open(DATA_DIR / "task4_metadata.json") as f:
    meta = json.load(f)
FEATURES, TARGET = meta["selected_features"], meta["target"]

train_df, val_df, test_df = (pd.read_csv(DATA_DIR / f"task4_{n}.csv") for n in ["train", "val", "test"])
X_train, y_train = train_df[FEATURES], train_df[TARGET]
y_val, y_test = val_df[TARGET].to_numpy(), test_df[TARGET].to_numpy()
sex_test = test_df["sex"].to_numpy()
print(f"test: {len(test_df):,} rows | {int(y_test.sum()):,} ever-smokers | {int((y_test == 0).sum()):,} never-smokers")

test: 98,454 rows | 38,594 ever-smokers | 59,860 never-smokers


In [2]:
# The five models with their final settings from Task 5 and Task 7
def task5_mlp(features):
    scale = [c for c in features if c != "sex"]   # sex is binary and not scaled, as in Task 5
    return Pipeline([
        ("preprocessor", ColumnTransformer([("continuous", StandardScaler(), scale)], remainder="passthrough")),
        ("classifier", MLPClassifier(hidden_layer_sizes=(16,), activation="relu", solver="adam", batch_size=512,
                                     max_iter=60, early_stopping=True, n_iter_no_change=8, random_state=42)),
    ])


models = {
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, random_state=42),
    "MLP (Task 5)": task5_mlp(FEATURES),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5, min_samples_leaf=50, max_features="sqrt",
                                            bootstrap=True, n_jobs=-1, random_state=18),
    "AdaBoost": AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1, random_state=18),
                                   n_estimators=50, learning_rate=1.0, random_state=18),
}
proba = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    proba[name] = (m.predict_proba(val_df[FEATURES])[:, 1], m.predict_proba(test_df[FEATURES])[:, 1])
    print(f"{name:14s} test AUC {roc_auc_score(y_test, proba[name][1]):.4f}")

Naive Bayes    test AUC 0.8495
Decision Tree  test AUC 0.8719
MLP (Task 5)   test AUC 0.8813
Random Forest  test AUC 0.8700
AdaBoost       test AUC 0.8748


In [3]:
# Reference models: the sex-only rule, and the Task 5 MLP retrained on demographic and body-size features only
SUBSETS = {
    "MLP on sex and age": ["sex", "age"],
    "MLP on sex, age and body size": ["sex", "age", "height", "weight", "waistline", "BMI", "waist_height_ratio"],
}
reference = {"Sex-only rule": (val_df["sex"].to_numpy(float), test_df["sex"].to_numpy(float))}
for name, feats in SUBSETS.items():
    m = task5_mlp(feats).fit(train_df[feats], y_train)
    reference[name] = (m.predict_proba(val_df[feats])[:, 1], m.predict_proba(test_df[feats])[:, 1])

# DNN: the five seeds of the selected configuration from Task 9, section 11
dnn_file = np.load(DATA_DIR / "task9_robustness_probabilities.npz")
DNN_SEEDS = [42, 1, 2, 3, 4]
dnn_runs = [(dnn_file[f"t19_selected_s{s}_val"], dnn_file[f"t19_selected_s{s}_test"]) for s in DNN_SEEDS]

In [4]:
# Matched contact budgets and a common recall requirement (Table 21)
BUDGETS = [0.10, 0.20, 0.30, 0.40]
TARGET_RECALL = 0.90


def found_in_budget(y, p, share):
    # Ever-smokers among the top `share` of people ranked by probability (ties at the cutoff averaged)
    k = int(round(share * len(y)))
    cutoff = np.sort(p)[::-1][k - 1]
    above, tied = p > cutoff, p == cutoff
    return y[above].sum() + (k - above.sum()) * y[tied].mean()


def at_common_recall(pv, pt):
    # Threshold chosen on validation as the highest one that reaches the target recall
    pos = np.sort(pv[y_val == 1])[::-1]
    thr = pos[int(np.ceil(TARGET_RECALL * len(pos))) - 1]
    pred = pt >= thr
    tp, fp = (pred & (y_test == 1)).sum(), (pred & (y_test == 0)).sum()
    return {"test recall": tp / (y_test == 1).sum(), "precision": tp / pred.sum(), "false positives": fp}


def budget_row(pv, pt, with_recall=True):
    row = {f"{int(b * 100)}%": found_in_budget(y_test, pt, b) for b in BUDGETS}
    if with_recall:
        row.update(at_common_recall(pv, pt))
    return row


rows = {"Sex-only rule": budget_row(*reference["Sex-only rule"], with_recall=False)}
rows.update({name: budget_row(*reference[name]) for name in SUBSETS})
rows.update({name: budget_row(*p) for name, p in proba.items()})
rows["DNN (mean of 5 seeds)"] = pd.DataFrame([budget_row(*r) for r in dnn_runs]).mean().to_dict()
budget_table = pd.DataFrame(rows).T
print("People contacted:", {f"{int(b * 100)}%": int(round(b * len(y_test))) for b in BUDGETS})
budget_table.round(3)

People contacted: {'10%': 9845, '20%': 19691, '30%': 29536, '40%': 39382}


,10%,20%,30%,40%,test recall,precision,false positives
Sex-only rule,6803.905,13608.501,20412.406,27217.003,NaN,NaN,NaN
MLP on sex and age,7532.847,14974.160,22060.004,28807.631,0.905,0.697,15173.0
"MLP on sex, age and body size",7700.250,15185.000,22271.333,28900.600,0.898,0.701,14802.0
Naive Bayes,7671.000,14803.000,21602.000,27861.000,0.898,0.682,16176.0
Decision Tree,7928.224,15434.254,22480.820,28943.336,0.903,0.702,14785.0
MLP (Task 5),8379.000,15877.000,22902.000,29358.000,0.900,0.706,14434.0
Random Forest,7983.000,15412.000,22478.000,28798.000,0.898,0.698,14991.0
AdaBoost,8224.572,15726.043,22651.000,29003.000,0.899,0.700,14849.0
DNN (mean of 5 seeds),8363.400,15945.800,22966.800,29394.600,0.900,0.707,14373.8


In [5]:
# AUC overall and within each sex (Table 24)
def auc_row(pt):
    return {"AUC": roc_auc_score(y_test, pt),
            "AUC among men": roc_auc_score(y_test[sex_test == 1], pt[sex_test == 1]),
            "AUC among women": roc_auc_score(y_test[sex_test == 0], pt[sex_test == 0])}


auc_rows = {name: auc_row(p[1]) for name, p in {**reference, **proba}.items()}
auc_rows["DNN (mean of 5 seeds)"] = pd.DataFrame([auc_row(r[1]) for r in dnn_runs]).mean().to_dict()
print(f"men: {(sex_test == 1).sum():,} ({y_test[sex_test == 1].mean():.1%} ever) | "
      f"women: {(sex_test == 0).sum():,} ({y_test[sex_test == 0].mean():.1%} ever)")
pd.DataFrame(auc_rows).T.round(4)

men: 51,959 (69.1% ever) | women: 46,495 (5.8% ever)


,AUC,AUC among men,AUC among women
Sex-only rule,0.8312,0.5000,0.5000
MLP on sex and age,0.8627,0.5994,0.6318
"MLP on sex, age and body size",0.8668,0.6119,0.6520
Naive Bayes,0.8495,0.5751,0.5556
Decision Tree,0.8719,0.6276,0.6746
MLP (Task 5),0.8813,0.6590,0.7058
Random Forest,0.8700,0.6235,0.6574
AdaBoost,0.8748,0.6402,0.6697
DNN (mean of 5 seeds),0.8825,0.6620,0.7147


In [6]:
# Prediction time for all models on the same CPU (Table 23)
# Boundary: from the test data in memory to predicted probabilities, including any scaling;
# reading files is excluded. One warm-up run, then ten timed repeats.
# The DNN is timed with the selected architecture; prediction time does not depend on the weight values.
torch.manual_seed(42)
layers, width = [], len(FEATURES)
for _ in range(2):
    layers += [nn.Linear(width, 509), nn.BatchNorm1d(509), nn.ReLU(), nn.Dropout(0.307)]
    width = 509
dnn_cpu = nn.Sequential(*layers, nn.Linear(width, 1)).eval()
dnn_scale_cols = meta["scaling"]["columns"]
dnn_scaler = StandardScaler().fit(train_df[dnn_scale_cols])
X_test = test_df[FEATURES]


def dnn_predict(X):
    Xs = X.copy()
    Xs[dnn_scale_cols] = dnn_scaler.transform(Xs[dnn_scale_cols])
    t = torch.from_numpy(Xs.to_numpy(np.float32))
    with torch.no_grad():
        return torch.sigmoid(torch.cat([dnn_cpu(t[i:i + 16_384]).squeeze(1) for i in range(0, len(t), 16_384)])).numpy()


predictors = {name: (lambda m=m: m.predict_proba(X_test)[:, 1]) for name, m in models.items()}
predictors["DNN (Task 9, on CPU)"] = lambda: dnn_predict(X_test)

timing = {}
for name, predict in predictors.items():
    predict()   # warm-up
    times = []
    for _ in range(10):
        t0 = time.perf_counter()
        predict()
        times.append(time.perf_counter() - t0)
    ms_per_10k = np.array(times) / len(X_test) * 10_000 * 1000
    timing[name] = {"median ms per 10k": np.median(ms_per_10k), "min": ms_per_10k.min(), "max": ms_per_10k.max()}

print(f"CPU cores: {os.cpu_count()} | torch threads: {torch.get_num_threads()}")
pd.DataFrame(timing).T.round(2)

CPU cores: 8 | torch threads: 4


,median ms per 10k,min,max
Naive Bayes,1.40,1.35,1.54
Decision Tree,0.55,0.54,0.56
MLP (Task 5),1.11,1.05,1.36
Random Forest,8.04,7.75,8.49
AdaBoost,16.71,16.44,17.21
"DNN (Task 9, on CPU)",18.78,18.73,19.10
